# Data overview

A look at what the data stage produced: the cleaned FreshRetailNet-50K data, `splits.parquet`, `eval_masks.parquet`, one PyPOTS training sample, and `demand_daily` once an imputation run exists.

**Before running:** do Steps 1–2 of `src/uncertainty_aware_planner/data/README.md` (`load`, `splits`, `masks`). Pick the project's `.venv` as the kernel, then Run All. Loading needs about 15 s and 5 GB of memory.

Gray marks show all data; colors always mean the same split: train blue, val orange, calib aqua, test yellow.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display
from plotly.subplots import make_subplots

ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()
)
sys.path.insert(0, str(ROOT / "src"))

from uncertainty_aware_planner.data.config import load_config, resolve  # noqa: E402
from uncertainty_aware_planner.data.cube import load_cube  # noqa: E402
from uncertainty_aware_planner.data.pypots_arrays import build_pypots_set  # noqa: E402

SPLIT_COLORS = {
    "train": "#2a78d6",
    "val": "#eb6834",
    "calib": "#1baf7a",
    "test": "#eda100",
}
ALL_DATA, IN_STOCK, MUTED, GRID = "#52514e", "#e1e0d9", "#898781", "#e1e0d9"
SALES_SCALE = ["#cde2fb", "#86b6ef", "#3987e5", "#256abf", "#104281", "#0d366b"]
axis = dict(
    gridcolor=GRID, linecolor="#c3c2b7", zeroline=False, tickfont=dict(color=MUTED)
)
pio.templates["data_overview"] = go.layout.Template(
    layout=dict(
        font=dict(
            family="system-ui, -apple-system, Segoe UI, sans-serif", color="#0b0b0b"
        ),
        paper_bgcolor="#fcfcfb",
        plot_bgcolor="#fcfcfb",
        xaxis=axis,
        yaxis=axis,
        margin=dict(l=60, r=30, t=70, b=50),
    )
)
pio.templates.default = "plotly_white+data_overview"
pio.renderers.default = "plotly_mimetype+notebook_connected"
pd.set_option("display.max_columns", 30)

## 1. Load the cleaned data and the parquet files

In [2]:
cfg = load_config()
processed = resolve(cfg["paths"]["processed_dir"])
cube = load_cube(cfg)  # validated and cleaned, same code every model uses
splits = pd.read_parquet(processed / "splits.parquet")
masks = pd.read_parquet(processed / "eval_masks.parquet")
dates = pd.to_datetime(cube.dates)

print(
    f"{cube.n_series:,} series x {cube.n_days} days ({cube.dates[0]} to {cube.dates[-1]})"
)
print("discount cleaning:", cube.cleaning_report)
pd.DataFrame(
    [
        {"file": str(p.relative_to(processed)), "MB": round(p.stat().st_size / 1e6, 2)}
        for p in sorted(processed.rglob("*.parquet"))
    ]
)

50,000 series x 97 days (2024-03-28 to 2024-07-02)
discount cleaning: {'train': {'discount_missing_code': 16039, 'discount_above_max': 34}, 'eval': {'discount_missing_code': 1300, 'discount_above_max': 0}}


,file,MB
0,eval_masks.parquet,11.87
1,splits.parquet,0.00


## 2. What one row of data looks like

One row per store-product-day (50,000 series × 97 days). Each day also carries 24 hourly sales values and 24 stock flags (1 = out of stock).

In [3]:
cube.daily.head(10)

,city_id,store_id,management_group_id,first_category_id,second_category_id,third_category_id,product_id,date,sale_amount,stock_hour6_22_cnt,discount,holiday_flag,activity_flag,precpt,avg_temperature,avg_humidity,avg_wind_level
0,0,0,2,29,78,82,4,2024-03-28,0.5,13,0.882,0,1,1.6999,15.48,73.54,1.97
1,0,0,2,29,78,82,4,2024-03-29,1.3,5,0.882,0,1,3.0190,15.08,76.56,1.71
2,0,0,2,29,78,82,4,2024-03-30,5.3,0,0.882,1,1,2.0942,15.91,76.47,1.73
3,0,0,2,29,78,82,4,2024-03-31,4.2,0,0.879,1,1,1.5618,16.13,77.40,1.76
4,0,0,2,29,78,82,4,2024-04-01,0.7,0,0.882,0,1,3.5386,15.37,78.26,1.25
5,0,0,2,29,78,82,4,2024-04-02,2.3,0,0.882,0,1,3.1459,15.69,76.63,2.13
6,0,0,2,29,78,82,4,2024-04-03,1.3,5,0.877,0,1,1.7165,16.11,76.31,1.51
7,0,0,2,29,78,82,4,2024-04-04,4.7,0,0.879,1,1,1.3021,16.08,74.24,1.47
8,0,0,2,29,78,82,4,2024-04-05,3.1,1,0.876,1,1,1.9090,16.24,71.99,1.46
9,0,0,2,29,78,82,4,2024-04-06,5.5,3,0.876,1,1,2.4001,16.72,78.42,1.67


In [4]:
first_day = pd.DataFrame(
    {
        "hour": range(24),
        "sales": cube.sales[0, 0],
        "out_of_stock": cube.stockout[0, 0],
    }
)
print(
    f"store {cube.series.store_id[0]}, product {cube.series.product_id[0]}, {cube.dates[0]}"
)
first_day.T

store 0, product 4, 2024-03-28


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23
hour,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23
sales,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
out_of_stock,True,True,True,False,False,False,False,False,False,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True


## 3. Splits (`splits.parquet`)

In [5]:
display(splits)

timeline = splits.assign(
    window=np.where(
        splits["fold"].isna(), "final", "fold " + splits["fold"].astype(str)
    ),
    start=pd.to_datetime(splits["start_date"]),
    end=pd.to_datetime(splits["end_date"]) + pd.Timedelta(days=1),
)
fig = px.timeline(
    timeline,
    x_start="start",
    x_end="end",
    y="window",
    color="split",
    color_discrete_map=SPLIT_COLORS,
    category_orders={"split": list(SPLIT_COLORS)},
    text="split",
    hover_data={"start_date": True, "end_date": True, "start": False, "end": False},
    title="Every fold trains on 4 weeks and checks on the next 4; calib and test are never in a fold",
)
fig.update_yaxes(
    categoryorder="array", categoryarray=timeline["window"].unique()[::-1], title=None
)
fig.update_traces(
    marker_line_color="#fcfcfb", marker_line_width=2, textposition="inside"
)
fig.update_layout(
    height=380,
    legend_title_text=None,
    xaxis_range=[dates[0], dates[-1] + pd.Timedelta(days=1)],
)
fig.show()

,fold,split,start_date,end_date
0,<NA>,train,2024-03-28,2024-06-18
1,<NA>,calib,2024-06-19,2024-06-25
2,<NA>,test,2024-06-26,2024-07-02
3,1,train,2024-04-01,2024-04-28
4,1,val,2024-04-29,2024-05-26
5,2,train,2024-04-04,2024-05-01
6,2,val,2024-05-02,2024-05-29
7,3,train,2024-04-08,2024-05-05
8,3,val,2024-05-06,2024-06-02
9,4,train,2024-04-10,2024-05-07


## 4. All 50,000 series over time

In [6]:
overview = pd.DataFrame(
    {
        "date": dates,
        "total_sales": cube.sales.sum(axis=(0, 2)),
        "share_of_hours_out_of_stock": cube.stockout.mean(axis=(0, 2)),
    }
)
display(overview.tail(14))

fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.1,
    subplot_titles=(
        "Total sales per day (scaled units)",
        "Share of hours out of stock",
    ),
)
for row, column in ((1, "total_sales"), (2, "share_of_hours_out_of_stock")):
    fig.add_scatter(
        x=overview["date"],
        y=overview[column],
        mode="lines",
        line=dict(color=ALL_DATA, width=2),
        showlegend=False,
        row=row,
        col=1,
    )
final = splits[splits["fold"].isna()].set_index("split")
for split in ("calib", "test"):
    start = pd.Timestamp(final.loc[split, "start_date"])
    end = pd.Timestamp(final.loc[split, "end_date"]) + pd.Timedelta(days=1)
    fig.add_vrect(
        x0=start, x1=end, fillcolor=SPLIT_COLORS[split], opacity=0.15, line_width=0
    )
    fig.add_annotation(
        x=start + (end - start) / 2, y=1.07, yref="paper", text=split, showarrow=False
    )
fig.update_yaxes(tickformat=".0%", row=2, col=1)
fig.update_layout(
    height=520, title="Sales and stockouts across the 97 days", margin_t=110
)
fig.show()

,date,total_sales,share_of_hours_out_of_stock
83,2024-06-19,55358.983,0.250728
84,2024-06-20,53356.762,0.228577
85,2024-06-21,58356.629,0.198564
86,2024-06-22,70919.014,0.224344
87,2024-06-23,70770.293,0.264221
88,2024-06-24,51491.231,0.251201
89,2024-06-25,58473.343,0.250376
90,2024-06-26,54976.997,0.229259
91,2024-06-27,55412.544,0.206433
92,2024-06-28,55461.006,0.180132


In [7]:
train_days = slice(0, cfg["expected"]["train_days"])
by_hour = pd.DataFrame(
    {
        "hour": range(24),
        "share_out_of_stock": cube.stockout[:, train_days].mean(axis=(0, 1)),
        "mean_sales": cube.sales[:, train_days].mean(axis=(0, 1)),
    }
)
display(by_hour.T.round(3))

fig = px.bar(
    by_hour,
    x="hour",
    y="share_out_of_stock",
    title="Night hours are flagged out of stock far more often (training days, all series)",
)
fig.update_traces(
    marker_color=ALL_DATA, marker_line_color="#fcfcfb", marker_line_width=2
)
fig.update_layout(yaxis_tickformat=".0%", yaxis_title=None, xaxis_dtick=2, height=380)
fig.show()

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23
hour,0.000,1.000,2.000,3.000,4.000,5.000,6.000,7.00,8.000,9.000,10.000,11.000,12.000,13.000,14.000,15.000,16.000,17.000,18.000,19.000,20.000,21.000,22.000,23.000
share_out_of_stock,0.411,0.400,0.375,0.329,0.267,0.174,0.062,0.05,0.057,0.071,0.094,0.123,0.148,0.166,0.180,0.199,0.240,0.293,0.336,0.364,0.387,0.410,0.419,0.418
mean_sales,0.008,0.005,0.003,0.002,0.003,0.010,0.031,0.06,0.093,0.107,0.099,0.071,0.054,0.050,0.059,0.076,0.081,0.059,0.039,0.030,0.025,0.016,0.010,0.006


## 5. One series up close

By default this picks a typical series: the median number of hidden hours in `FOLD`'s val window, among series that also have hidden hours in calib. Set `SERIES = (store_id, product_id)` to look at another one.

In [8]:
FOLD = 1
SERIES = None

fold_masks = masks[masks["fold"].eq(FOLD).fillna(False)]
calib_masks = masks[masks["split"] == "calib"]
if SERIES is None:
    counts = fold_masks.groupby(["store_id", "product_id"]).size()
    in_calib = calib_masks.groupby(["store_id", "product_id"]).size().index
    counts = counts[counts.index.isin(in_calib)].sort_values()
    SERIES = counts.index[len(counts) // 2]
store_id, product_id = SERIES
i = int(cube.series_index([store_id], [product_id])[0])


def hidden_grid(rows):
    """(24, n_days) bool: hours of this series hidden by the given mask rows."""
    rows = rows[(rows["store_id"] == store_id) & (rows["product_id"] == product_id)]
    grid = np.zeros((24, cube.n_days), dtype=bool)
    day = (rows["datetime"].dt.normalize() - dates[0]).dt.days.to_numpy()
    grid[rows["datetime"].dt.hour.to_numpy(), day] = True
    return grid


sales = cube.sales[i].T  # (24 hours, n_days)
stockout = cube.stockout[i].T
status = np.where(stockout, 1, 0)
status[hidden_grid(fold_masks)] = 2
status[hidden_grid(calib_masks)] = 3

per_day = cube.daily.iloc[i * cube.n_days : (i + 1) * cube.n_days][
    ["date", "sale_amount", "stock_hour6_22_cnt", "discount", "holiday_flag"]
].assign(
    hours_out_of_stock=stockout.sum(axis=0),
    hidden_fold_val=(status == 2).sum(axis=0),
    hidden_calib=(status == 3).sum(axis=0),
)
print(f"store {store_id}, product {product_id}")
display(per_day.reset_index(drop=True))

store 172, product 596


,date,sale_amount,stock_hour6_22_cnt,discount,holiday_flag,hours_out_of_stock,hidden_fold_val,hidden_calib
0,2024-03-28,1.2,0,0.967,0,3,0,0
1,2024-03-29,1.1,6,0.920,0,10,0,0
2,2024-03-30,1.3,9,0.870,1,14,0,0
3,2024-03-31,1.3,8,0.882,1,13,0,0
4,2024-04-01,0.3,5,1.000,0,9,0,0
...,...,...,...,...,...,...,...,...
92,2024-06-28,1.0,5,1.000,0,7,0,0
93,2024-06-29,1.1,4,1.000,1,8,0,0
94,2024-06-30,1.0,6,1.000,1,10,0,0
95,2024-07-01,1.0,0,1.000,0,0,0,0


In [9]:
labels = np.array(
    [
        "in stock",
        "out of stock (imputed)",
        f"hidden in fold {FOLD} val (scored)",
        "hidden in calib (scored)",
    ]
)
colors = [IN_STOCK, ALL_DATA, SPLIT_COLORS["val"], SPLIT_COLORS["calib"]]

fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.12,
    subplot_titles=(
        "Hourly sales (blank = out of stock, true demand unknown)",
        "What each hour is used for",
    ),
)
fig.add_heatmap(
    x=dates,
    y=list(range(24)),
    z=np.where(stockout, np.nan, sales),
    colorscale=SALES_SCALE,
    colorbar=dict(title="sales", len=0.42, y=0.79),
    xgap=1,
    ygap=1,
    hovertemplate="%{x|%b %d}, hour %{y}<br>sales %{z:.3f}<extra></extra>",
    row=1,
    col=1,
)
fig.add_heatmap(
    x=dates,
    y=list(range(24)),
    z=status,
    text=labels[status],
    zmin=-0.5,
    zmax=3.5,
    colorscale=[
        [k / 4 if j == 0 else (k + 1) / 4, c]
        for k, c in enumerate(colors)
        for j in (0, 1)
    ],
    colorbar=dict(tickvals=[0, 1, 2, 3], ticktext=list(labels), len=0.42, y=0.21),
    xgap=1,
    ygap=1,
    hovertemplate="%{x|%b %d}, hour %{y}<br>%{text}<extra></extra>",
    row=2,
    col=1,
)
fig.update_yaxes(title="hour", dtick=6, autorange="reversed", showgrid=False)
fig.update_xaxes(showgrid=False)
fig.update_layout(height=640, title=f"Store {store_id}, product {product_id}")
fig.show()

## 6. Evaluation masks (`eval_masks.parquet`)

In [10]:
display(masks.head())
per_window = (
    masks.groupby(["split", "fold"], dropna=False)
    .size()
    .rename("hidden_hours")
    .reset_index()
    .assign(
        window=lambda d: np.where(
            d["fold"].isna(), "calib", "fold " + d["fold"].astype(str) + " val"
        )
    )
)
display(per_window)

fig = px.bar(
    per_window,
    x="window",
    y="hidden_hours",
    color="split",
    color_discrete_map=SPLIT_COLORS,
    text_auto=".3s",
    title="Hidden in-stock hours per window (calib covers 1 week, each val window 4)",
)
fig.update_traces(
    marker_line_color="#fcfcfb", marker_line_width=2, textposition="outside"
)
fig.update_layout(height=380, xaxis_title=None, yaxis_title=None, showlegend=False)
fig.show()

,store_id,product_id,datetime,split,fold,mask_seed
0,0,4,2024-05-08 07:00:00,val,1,1024
1,0,4,2024-05-08 08:00:00,val,1,1024
2,0,4,2024-05-08 09:00:00,val,1,1024
3,0,4,2024-05-08 10:00:00,val,1,1024
4,0,4,2024-05-08 11:00:00,val,1,1024


,split,fold,hidden_hours,window
0,calib,<NA>,788600,calib
1,val,1,3193482,fold 1 val
2,val,2,3199644,fold 2 val
3,val,3,3338264,fold 3 val
4,val,4,3431991,fold 4 val
5,val,5,3399760,fold 5 val
6,val,6,3425440,fold 6 val


In [11]:
hidden_by_hour = (
    masks["datetime"].dt.hour.value_counts().sort_index().rename_axis("hour")
)
hidden_by_hour = hidden_by_hour.rename("hidden_hours").reset_index()
display(hidden_by_hour.T)

fig = px.bar(
    hidden_by_hour,
    x="hour",
    y="hidden_hours",
    title="Hidden hours cover all 24 hours of the day, most often midday",
)
fig.update_traces(
    marker_color=ALL_DATA, marker_line_color="#fcfcfb", marker_line_width=2
)
fig.update_layout(height=380, xaxis_dtick=2, yaxis_title=None)
fig.show()

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23
hour,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23
hidden_hours,668999,703996,739213,773840,808613,843743,878603,913607,948518,983206,983487,983798,983780,983912,984140,984008,949118,914089,879036,843840,808967,773964,738946,703758


## 7. One PyPOTS sample (what the imputation models see)

The same series in fold `FOLD`'s val window: 672 hourly steps × 9 features (sales, the 7 covariates, hour of day). `X` has real stockouts and the hidden hours as NaN; `X_ori` keeps the true sales at the hidden hours.

In [12]:
val_set, val_index = build_pypots_set(cube, cfg, "val", fold=FOLD, masks=masks)
row = int(
    np.flatnonzero(
        (val_index["store_id"] == store_id) & (val_index["product_id"] == product_id)
    )[0]
)
x, x_ori = val_set["X"][row], val_set["X_ori"][row]
del val_set  # about 0.8 GB

feature_names = ["sales", *cfg["imputation"]["covariates"], "hour"]
steps = pd.date_range(val_index.loc[row, "start_date"], periods=len(x), freq="h")
sample = pd.DataFrame(x, index=steps, columns=feature_names).assign(
    true_sales=x_ori[:, 0]
)
print(f"sample shape {x.shape}; window starts {val_index.loc[row, 'start_date']}")
display(sample.head(24))

sample shape (672, 9); window starts 2024-04-29


,sales,discount,holiday_flag,activity_flag,precpt,avg_temperature,avg_humidity,avg_wind_level,hour,true_sales
2024-04-29 00:00:00,0.0,0.847273,0.0,0.0,0.717679,0.816121,0.998694,0.654135,0.000000,0.0
2024-04-29 01:00:00,0.0,0.847273,0.0,0.0,0.717679,0.816121,0.998694,0.654135,0.043478,0.0
2024-04-29 02:00:00,0.0,0.847273,0.0,0.0,0.717679,0.816121,0.998694,0.654135,0.086957,0.0
2024-04-29 03:00:00,0.0,0.847273,0.0,0.0,0.717679,0.816121,0.998694,0.654135,0.130435,0.0
2024-04-29 04:00:00,0.0,0.847273,0.0,0.0,0.717679,0.816121,0.998694,0.654135,0.173913,0.0
2024-04-29 05:00:00,0.0,0.847273,0.0,0.0,0.717679,0.816121,0.998694,0.654135,0.217391,0.0
2024-04-29 06:00:00,0.1,0.847273,0.0,0.0,0.717679,0.816121,0.998694,0.654135,0.260870,0.1
2024-04-29 07:00:00,0.1,0.847273,0.0,0.0,0.717679,0.816121,0.998694,0.654135,0.304348,0.1
2024-04-29 08:00:00,0.2,0.847273,0.0,0.0,0.717679,0.816121,0.998694,0.654135,0.347826,0.2
2024-04-29 09:00:00,0.1,0.847273,0.0,0.0,0.717679,0.816121,0.998694,0.654135,0.391304,0.1


In [13]:
hidden = np.isnan(x[:, 0]) & ~np.isnan(x_ori[:, 0])
real_stockout = np.isnan(x_ori[:, 0])
rug = -0.06 * np.nanmax(x_ori[:, 0])

fig = go.Figure()
fig.add_scatter(
    x=steps,
    y=x[:, 0],
    mode="lines",
    line=dict(color=ALL_DATA, width=1.5),
    name="sales the model sees (X)",
)
fig.add_scatter(
    x=steps,
    y=np.where(hidden, x_ori[:, 0], np.nan),
    mode="lines",
    line=dict(color=SPLIT_COLORS["val"], width=2),
    name="hidden sales, scored (X_ori)",
)
fig.add_scatter(
    x=steps[real_stockout],
    y=np.full(real_stockout.sum(), rug),
    mode="markers",
    marker=dict(color=ALL_DATA, symbol="line-ns-open", size=10),
    name="real stockout, model imputes",
)
fig.update_layout(
    height=420,
    title=f"Store {store_id}, product {product_id}: fold {FOLD} val input",
    legend=dict(orientation="h", y=1.02, yanchor="bottom", x=0),
    margin_t=110,
    yaxis_title="sales",
)
fig.show()

## 8. `demand_daily` (after an imputation run)

Built by Step 4 of the data README from an `imputed_hourly` file. Shows the newest file if one exists.

In [14]:
runs = sorted(
    (processed / "demand_daily").glob("*.parquet"), key=lambda p: p.stat().st_mtime
)
if runs:
    demand = pd.read_parquet(runs[-1])
    print(runs[-1].name, demand.shape)
    display(demand.head(10))
else:
    print(
        "No demand_daily file yet: it is built in Step 4, after the first imputation run."
    )

No demand_daily file yet: it is built in Step 4, after the first imputation run.
